In [1]:
# A set of functions to calculate combinations, necessary for creating complete sets of operators for
# constructing hierarchies of equations of motions
using Combinatorics
using Base.Threads
include("operator_terms.jl")
include("diff_Eq.jl")
include("print_terms.jl")

fprintln (generic function with 1 method)

In [2]:
function mrange(lengths::Vector{Int}; startpoint::Int=1)
    # computes the following type of nested loops in a single loop
    # for i in startpoint:lengths[1]
    #     for j in startpoint:lengths[2]
    #         # n times
    #         ...
    #     end
    # end
    curr_index::Vector{Int} = ones(Int, length(lengths)) * startpoint
    max_iterations = prod(lengths .+ (1 - startpoint))
    counter = 0
    chnl = Channel() do channel
        while counter < max_iterations
            put!(channel, copy(curr_index))
            counter += 1
            for i in length(lengths):-1:startpoint
                curr_index[i] += 1
                if curr_index[i] > lengths[i]
                    curr_index[i] = startpoint
                    if i == 1
                        return
                    end
                else
                    break
                end
            end
        end
    end
    return chnl
end

function mgreater_range(max_val::Int, n::Int; startpoint::Int=1, offdiag::Int=0)
    # computes the following type of nested loops in a single loop
    # for i in startpoint:max_val
    #     for j in (i+offdiag):max_val
    #         # n times
    #         ...
    #     end
    # end
    curr_index::Vector{Int} = Vector{Int}(undef, n)
    curr_index[1] = startpoint
    for i in 2:n
        curr_index[i] = curr_index[i-1] + offdiag
    end
    #curr_index = curr_index[end:-1:1]
    max_vals::Vector{Int} = Vector{Int}(undef, n)
    max_vals[1] = max_val
    for i in 2:n
        max_vals[i] = max_vals[i-1] - offdiag
    end
    max_vals = max_vals[end:-1:1]
    changed::Bool = false
    chnl = Channel() do channel
        put!(channel, copy(curr_index))
        # change last element 
        while true
            if curr_index[n] < max_vals[n]
                curr_index[n] += 1
                put!(channel, copy(curr_index))
            else
                changed = false
                for i in (n-1):-1:1
                    if curr_index[i] + 1 <= max_vals[i]
                        curr_index[i] += 1
                        for j in (i+1):n
                            curr_index[j] = curr_index[j-1] + offdiag
                        end
                        put!(channel, copy(curr_index))
                        changed = true
                        break
                    end
                end
                if !changed
                    # last element is max_val but should be changed
                    return
                end
            end
        end
    end
    return chnl
end

mgreater_range (generic function with 1 method)

In [3]:
# function to generate all combinations of numbers from 1 to max_num with length n
function all_int_combinations(max_num::Int, n::Int)
    # uses mrange
    max_nums::Vector{Int} = max_num * ones(Int, n)
    all_combinations::Vector{Vector{Int}} = collect(mrange(max_nums))
    return all_combinations
end

# A function that constructs all combinations of increasing or equal numbers from 1 to max_num with length n
function geq_int_combinations(max_num::Int, n::Int)
    # uses mgreater_range
    all_combinations::Vector{Vector{Int}} = collect(mgreater_range(max_num, n, offdiag=0))
    return all_combinations
end
## Test 
all_int_combinations(3, 2)
geq_int_combinations(3, 2)

6-element Vector{Vector{Int}}:
 [1, 1]
 [1, 2]
 [1, 3]
 [2, 2]
 [2, 3]
 [3, 3]

In [4]:
# A function that constructs a vector of all_int_combinations for each number of lengths from 1 to max_length
function all_int_combinations_upto(max_num::Int, max_length::Int)
    all_combinations::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}(undef, max_length)
    for i in 1:max_length
        all_combinations[i] = all_int_combinations(max_num, i)
    end
    return all_combinations
end

all_int_combinations_upto (generic function with 1 method)

In [5]:
function geq_spin_combinations(number_of_spins::Int)#::Vector{String}
    # all combinations of 'x,y,z' for a given number of spins with (greater equal ordering)
    spin_types::Array{String} = ["x", "y", "z"]
    ind_strings::Array{String} = ["i", "j", "k", "l", "m", "n", "o", "p", "q", "r"]

    if number_of_spins > 10
        error("Too many spins. Maximum 10 possible.")
    end
    
    comb::Array{String} = []
    if number_of_spins == 0
        # Add empty string
        push!(comb, "")
        return comb
    end
    
    ind_substring = ind_strings[1:number_of_spins]
    spin_ind_comb = geq_int_combinations(3, number_of_spins)
    for i in 1:length(spin_ind_comb)
        curr_comb = spin_ind_comb[i]
        curr_string = ""
        for j in 1:number_of_spins
            curr_string *= spin_types[curr_comb[j]] * ind_substring[j] 
            if j < number_of_spins
                curr_string *= "*"
            end
        end
        push!(comb, curr_string)
    end
    return comb
end
#test
geq_spin_combinations(2)

6-element Vector{String}:
 "xi*xj"
 "xi*yj"
 "xi*zj"
 "yi*yj"
 "yi*zj"
 "zi*zj"

In [6]:
function bosonic_combinations_of_order(n::Int)::Vector{String}
    # Only consider bosonic operators of type $(a^\dagger)^p a^q$ with $q >= p$ (fun fact: p=q is hermitian)
    all_comb::Array{String} = []
    for i in 0:n÷2
        push!(all_comb, "+"^i * "-"^(n-i))
    end
    return all_comb
end

# test
bosonic_combinations_of_order(4)

3-element Vector{String}:
 "----"
 "+---"
 "++--"

In [7]:
# generate all combinations of N vectors of vectors
function kron_indexes(args...) 
    # args is a tuple of vectors of vectors
    # e.g. A = [[1,2], [3,4]], B = [[5,6], [7,8]]
    # returns [[1,2,5,6], [1,2,7,8], [3,4,5,6], [3,4,7,8]]
    # check that every element in args is a vector of vectors (of some type)
    how_many_args = length(args)
    if how_many_args < 2
        error("Need at least two arguments")
    end

    for arg in args
        if !(typeof(arg) <: AbstractVector)
            error("All arguments must be vectors of vectors")
        elseif !(typeof(arg[1]) <: AbstractVector)
            error("All arguments must be vectors of vectors")
        end
    end
    # check if all vectors of vectors are of same type?
    # first first arguments subtypes
    subtype = typeof(args[1][1][1])
    for i in 2:how_many_args
        if !(subtype <: typeof(args[i][1][1]))
            error("All vectors of vectors must be of same type")
        end
    end
    lengths = [length(args[i]) for i in 1:how_many_args]
    sublengths = [length(args[i][1]) for i in 1:how_many_args]
    subvec = Vector{subtype}(undef, sum(sublengths))
    C = Vector{Vector{subtype}}(undef, prod(lengths))
    counter = 0
    for i_s in mrange(lengths)
        counter += 1
        start_ind = 1
        for j in 1:how_many_args
            end_ind = start_ind + sublengths[j] - 1
            subvec[start_ind:end_ind] = args[j][i_s[j]]
            start_ind = end_ind + 1
        end
        C[counter] = copy(subvec)
    end
    return C
end
# Test 
A = [[1,2], [3,4]]
B = [[5,6], [7,8]]
C = [[9,10], [11,12]]
kron_indexes(A, B, C)

8-element Vector{Vector{Int}}:
 [1, 2, 5, 6, 9, 10]
 [1, 2, 5, 6, 11, 12]
 [1, 2, 7, 8, 9, 10]
 [1, 2, 7, 8, 11, 12]
 [3, 4, 5, 6, 9, 10]
 [3, 4, 5, 6, 11, 12]
 [3, 4, 7, 8, 9, 10]
 [3, 4, 7, 8, 11, 12]

In [9]:
# Now added to operator_terms
mutable struct Indexed_Term # Term for within DE_Term_indexed and DE_Term_Multi_indexed, all strings replaced by indexes to apropriate vectors
    # Tuple{Int,Bool,Int,ComplexF64} - Tuple consists of (operator index, conjugate, variables index, coefficient)
    operator_index::Int
    conjugate::Bool
    variables_index::Int
    coefficient::ComplexF64
    function Indexed_Term(operator_index::Int, conjugate::Bool, variables_index::Int, coefficient::ComplexF64)
        new(operator_index, conjugate, variables_index, coefficient)
    end
end

mutable struct Abstract_Term # Abstract reduced representation of a term, 
    # Abstract terms are separated into x y and z operators to allow the easy calculation of indexing into a vector of operator expectation values
    # keeps references to variables in Term form 
    term_str::String
    spin_orders::Vector{Int}
    spin_indices::Vector{Vector{Int}} # hijk -> 0123 but separated into x y and z groups 
    coeff::ComplexF64
    vars::Vector{String}
    conjugate::Bool

    function Abstract_Term(term_str::String, spin_orders::Vector{Int}, spin_indices::Vector{Vector{Int}}, coeff::ComplexF64=0.0 + 0.0im, vars::Vector{String}=String[], conjugate::Bool=false)
        new(term_str, spin_orders, spin_indices, coeff, vars, conjugate)
    end
end

# Function that transforms a Term into an Abstract_Term separating the x, y and z operators into different groups
function term_to_abstract_term(term::Term, min_ind='h')::Abstract_Term
    # term is a Term from operator_terms.jl
    # returns a vector of how many of each operator is in the term counting the number of +, -, x, y and z operators
    # e.g. for the term +x*y*z*x*y*z the vector [0, 0, 2, 2, 2] is returned
    spin_orders::Vector{Int} = zeros(Int, 3)
    indices = term.spin_indices  # transform into numbers 
    inds = [Int(i) for i in indices]
    if typeof(min_ind) <: String
        min_ind_ind = Int(min_ind[1])
    elseif typeof(min_ind) <: Char
        min_ind_ind = Int(min_ind)
    else
        error("min_ind must be a string or a char")
    end
    inds = inds .- min_ind_ind # min ind is zero
    spin_indices::Vector{Vector{Int}} = Vector{Vector{Int}}(undef, 3)
    for i in 1:3
        spin_indices[i] = []
    end
    for (s, i) in zip(term.spin_types, inds)
        if s == 'x'
            spin_orders[1] += 1
            push!(spin_indices[1], i)
        elseif s == 'y'
            spin_orders[2] += 1
            push!(spin_indices[2], i)
        elseif s == 'z'
            spin_orders[3] += 1
            push!(spin_indices[3], i)
        else
            error("spin type must be x, y or z, got $s")
        end
    end
    # transform operator into a string 
    term_str, _, _ = term2reducedstr(term)
    return Abstract_Term(term_str, spin_orders, spin_indices, term.coeff, term.vars, term.conjugate)
end
# Test 
term = make_term("++-yn*zm*xi*yj*xk*zl")
display(term)
term_to_abstract_term(term)

 a†²axᵢyⱼxₖzₗzₘyₙ 

Abstract_Term("++-xxyyzz", [2, 2, 2], [[1, 3], [2, 6], [4, 5]], 1.0 + 0.0im, String[], false)

In [ ]:
# A function that transforms an Abstract_Index_Term into a Vector of Index_Terms with a specified indexing
function abstract_index_term_to_index_term(abstract_index_term::Abstract_Intdex_Term, indexes::Vector{Int})::Index_Term
    how_many_each = abstract_index_term.how_many_each
    operator_indexes = abstract_index_term.operator_indexes
    new_indexes = indexes[operator_indexes]
    return Index_Term(how_many_each, new_indexes)
end

# A function that constructs all indexes of an abstract_term 

In [41]:
#### Now generate operator sets (array of strings) for single spin systems: 

In [43]:
function single_spin_gen_operator_strings_of_order(n::Int)::Vector{String}
    operators::Vector{String} = []
    pauli::Vector{String} = ["x", "y", "z"]

    # All combinations of bosonic operators of order n
    all_comb_n = bosonic_combinations_of_order(n)
    append!(operators, all_comb_n)

    # All combinations of bosonic operators of order n-1, appended with each Pauli operator
    all_comb_n_1 = bosonic_combinations_of_order(n-1)
    for p in pauli
        append!(operators, [comb * p for comb in all_comb_n_1])
    end

    return operators
end
# test
single_spin_gen_operator_strings_of_order(3)

8-element Vector{String}:
 "---"
 "+--"
 "--x"
 "+-x"
 "--y"
 "+-y"
 "--z"
 "+-z"

In [44]:
function single_spin_gen_operator_strings_up_to_order(n::Int; separate_orders::Bool=true)::Union{Vector{String}, Vector{Vector{String}}}
    operators::Any = separate_orders ? Vector{Vector{String}}() : Vector{String}() # = separate_orders ? Vector{String}[] : String[]
    if separate_orders
        for i in 1:n
            push!(operators, single_spin_gen_operator_strings_of_order(i))
        end
        return operators
    else
        for i in 1:n
            append!(operators, single_spin_gen_operator_strings_of_order(i))
        end
        return operators
    end
end
# test
display(single_spin_gen_operator_strings_up_to_order(3, separate_orders=true))
display(single_spin_gen_operator_strings_up_to_order(3, separate_orders=false))

3-element Vector{Vector{String}}:
 ["-", "x", "y", "z"]
 ["--", "+-", "-x", "-y", "-z"]
 ["---", "+--", "--x", "+-x", "--y", "+-y", "--z", "+-z"]

17-element Vector{String}:
 "-"
 "x"
 "y"
 "z"
 "--"
 "+-"
 "-x"
 "-y"
 "-z"
 "---"
 "+--"
 "--x"
 "+-x"
 "--y"
 "+-y"
 "--z"
 "+-z"

In [45]:
function single_spin_gen_operators_of_order(n::Int)::Vector{Term}
    # create operator strings
    operator_strings::Vector{String} = single_spin_gen_operator_strings_of_order(n)
    # create operators
    operators::Vector{Term} = []
    for op in operator_strings
        push!(operators, make_term(op))
    end
    return operators
end
# test
display(single_spin_gen_operators_of_order(3))

8-element Vector{Term}:
  aaa 
  a†aa 
  aax 
  a†ax 
  aay 
  a†ay 
  aaz 
  a†az 

In [46]:
function single_spin_gen_operators_up_to_order(n::Int)::Vector{Vector{Term}}
    # create operators
    operators::Vector{Vector{Term}} = [single_spin_gen_operators_of_order(i) for i in 1:n]
    return operators
end
# test
display(single_spin_gen_operators_up_to_order(3))

3-element Vector{Vector{Term}}:
 [Term("", "", "-", [1], 1.0 + 0.0im, String[]), Term("x", " ", "", Int[], 1.0 + 0.0im, String[]), Term("y", " ", "", Int[], 1.0 + 0.0im, String[]), Term("z", " ", "", Int[], 1.0 + 0.0im, String[])]
 [Term("", "", "--", [1, 1], 1.0 + 0.0im, String[]), Term("", "", "+-", [1, 1], 1.0 + 0.0im, String[]), Term("x", " ", "-", [1], 1.0 + 0.0im, String[]), Term("y", " ", "-", [1], 1.0 + 0.0im, String[]), Term("z", " ", "-", [1], 1.0 + 0.0im, String[])]
 [Term("", "", "---", [1, 1, 1], 1.0 + 0.0im, String[]), Term("", "", "+--", [1, 1, 1], 1.0 + 0.0im, String[]), Term("x", " ", "--", [1, 1], 1.0 + 0.0im, String[]), Term("x", " ", "+-", [1, 1], 1.0 + 0.0im, String[]), Term("y", " ", "--", [1, 1], 1.0 + 0.0im, String[]), Term("y", " ", "+-", [1, 1], 1.0 + 0.0im, String[]), Term("z", " ", "--", [1, 1], 1.0 + 0.0im, String[]), Term("z", " ", "+-", [1, 1], 1.0 + 0.0im, String[])]

In [47]:
function single_spin_gen_DE_of_order(n::Int)::Vector{DE_Term}
    # create operator strings
    operators::Vector{Term} = single_spin_gen_operators_of_order(n)
    # create DE operators
    DE_operators::Vector{DE_Term} = []
    for op in operators
        push!(DE_operators, time_derivative_expectation_of_operator(op))
    end
    return DE_operators
end
# test
display(single_spin_gen_DE_of_order(1))

4-element Vector{DE_Term}:
 d⟨a⟩/dt = √̅κβ - κ/2κ⟨a⟩ - g/2(i⟨x⟩ + ⟨y⟩)
 d⟨x⟩/dt = ig(⟨az⟩ - ⟨a†z⟩) - 2γγ⟨x⟩ - ΔΔ⟨y⟩ - Γ/2Γ⟨x⟩
 d⟨y⟩/dt =  - g(⟨az⟩ + ⟨a†z⟩) - 2γγ⟨y⟩ + ΔΔ⟨x⟩ - Γ/2Γ⟨y⟩
 d⟨z⟩/dt = g(⟨ay⟩ - i⟨ax⟩ + ⟨a†y⟩ + i⟨a†x⟩) - Γ(⟨z⟩ + 1)

In [48]:
function single_spin_gen_DE_up_to_order(n::Int)::Vector{Vector{DE_Term}}
    # create DE operators
    DE_operators::Vector{Vector{DE_Term}} = [single_spin_gen_DE_of_order(i) for i in 1:n]
    return DE_operators
end
# test
eq_of_eq = single_spin_gen_DE_up_to_order(3)
for eqs in eq_of_eq
    for eq in eqs
        display(eq)
    end
end

d⟨a⟩/dt = √̅κβ - κ/2κ⟨a⟩ - g/2(i⟨x⟩ + ⟨y⟩)

d⟨x⟩/dt = ig(⟨az⟩ - ⟨a†z⟩) - 2γγ⟨x⟩ - ΔΔ⟨y⟩ - Γ/2Γ⟨x⟩

d⟨y⟩/dt =  - g(⟨az⟩ + ⟨a†z⟩) - 2γγ⟨y⟩ + ΔΔ⟨x⟩ - Γ/2Γ⟨y⟩

d⟨z⟩/dt = g(⟨ay⟩ - i⟨ax⟩ + ⟨a†y⟩ + i⟨a†x⟩) - Γ(⟨z⟩ + 1)

d⟨aa⟩/dt = 2√̅κβ√̅κβ⟨a⟩ - κκ⟨a²⟩ - g(i⟨ax⟩ + ⟨ay⟩)

...

d⟨a†ax⟩/dt =  - Γ/2Γ⟨a†ax⟩ + √̅κβ√̅κβ⟨a†x⟩ - κκ⟨a†ax⟩ + ig/2(2⟨a†a²z⟩ - ⟨a†⟩ - 2⟨a†²az⟩ - ⟨a†z⟩ + ⟨a⟩ + ⟨az⟩) - 2γγ⟨a†ax⟩ - ΔΔ⟨a†ay⟩ + √̅κβ*√̅κβ*⟨ax⟩

d⟨aay⟩/dt = 2√̅κβ√̅κβ⟨ay⟩ - κκ⟨a²y⟩ - g(⟨a³z⟩ + ⟨a†a²z⟩ + ⟨az⟩ + ⟨a⟩) - 2γγ⟨a²y⟩ + ΔΔ⟨a²x⟩ - Γ/2Γ⟨a²y⟩

d⟨a†ay⟩/dt =  - Γ/2Γ⟨a†ay⟩ + √̅κβ√̅κβ⟨a†y⟩ - κκ⟨a†ay⟩ - g/2(2⟨a†a²z⟩ + 2⟨a†²az⟩ + ⟨a†z⟩ + ⟨a†⟩ + ⟨az⟩ + ⟨a⟩) - 2γγ⟨a†ay⟩ + ΔΔ⟨a†ax⟩ + √̅κβ*√̅κβ*⟨ay⟩

d⟨aaz⟩/dt = 2√̅κβ√̅κβ⟨az⟩ - κκ⟨a²z⟩ + g(⟨a³y⟩ - i⟨a³x⟩ + ⟨a†a²y⟩ + ⟨ay⟩ + i⟨a†a²x⟩ + i⟨ax⟩) - Γ(⟨a²z⟩ + ⟨a²⟩)

d⟨a†az⟩/dt = √̅κβ√̅κβ⟨a†z⟩ - κκ⟨a†az⟩ + g/2(2⟨a†a²y⟩ - 2i⟨a†a²x⟩ + 2⟨a†²ay⟩ + ⟨a†y⟩ + 2i⟨a†²ax⟩ + i⟨a†x⟩ + ⟨ay⟩ - i⟨ax⟩) + √̅κβ*√̅κβ*⟨az⟩ - Γ(⟨a†az⟩ + ⟨a†a⟩)

In [49]:
##### Multi Spin Systems #####

In [50]:
function multi_spin_gen_operator_strings_of_order(n::Int; max_spin::Int=-1, max_cavity::Int=-1, with_permutations::Bool=true)::Vector{String}
    # n is the order of operators,
    # max_spin is the maximum number of spins in the system (if negative, max is infinite)
    # max_cavity is the maximum number of bosonic operators in the system (if negative, max is infinite)
    # with_permutations: if true, return all permutations of the operators (xy and yx) of false only returns one order
    if max_spin < 0
        max_spin = n
    end
    if max_spin > 10
        error("Too many spins. Maximum 10 possible.")
    end
    if max_cavity < 0
        max_cavity = n
    end
    if max_cavity + max_spin < n
        error("Not enough operators to reach order n (max_spin + max_cavity < n).")
    end

    min_spins::Int = n-max_cavity
    if min_spins < 0
        min_spins = 0
    end

    operators::Array{String} = []
    # loop over cases of different numbers of spin operators
    for n_spins in min_spins:max_spin
        # loop over all combinations of spin operators
        spin_combinations = spin_combinations_strings(n_spins, with_permutations=with_permutations)
        # construct all combinations of n-n_spin cavity operators ( with p >= q )
        cavity_combinations = bosonic_combinations_of_order(n-n_spins)
        # append all combinations of spin and cavity operators
        for spin_comb in spin_combinations
            for cavity_comb in cavity_combinations
                push!(operators, spin_comb*cavity_comb)
            end
        end
    end
    return operators
end
# test
multi_spin_gen_operator_strings_of_order(2)#, max_spin=3, max_cavity=2, with_permutations=true)

14-element Vector{String}:
 "--"
 "+-"
 "xi-"
 "yi-"
 "zi-"
 "xi*xj"
 "xi*yj"
 "xi*zj"
 "yi*xj"
 "yi*yj"
 "yi*zj"
 "zi*xj"
 "zi*yj"
 "zi*zj"

In [51]:
function multi_spin_gen_operator_strings_up_to_order(n::Int; max_spin::Int=-1, max_cavity::Int=-1, separate_orders::Bool=true, with_permutations::Bool=true)::Union{Vector{String}, Vector{Vector{String}}}
    # n is the order of operators,
    # max_spin is the maximum number of spins in the system (if negative, max is infinite)
    # max_cavity is the maximum number of bosonic operators in the system (if negative, max is infinite)
    # separate_orders: if true, return operators separated by order, if false, return all operators in one array
    # with_permutations: if true, return all permutations of the operators (xy and yx) of false only returns one order
    operators::Any = separate_orders ? Vector{Vector{String}}() : Vector{String}() # = separate_orders ? Vector{String}[] : String[]
    if separate_orders
        for i in 1:n
            push!(operators, multi_spin_gen_operator_strings_of_order(i, max_spin=max_spin, max_cavity=max_cavity, with_permutations=with_permutations))
        end
    else
        for i in 1:n
            append!(operators, multi_spin_gen_operator_strings_of_order(i, max_spin=max_spin, max_cavity=max_cavity, with_permutations=with_permutations))
        end
    end
    return operators
end
# test
display(multi_spin_gen_operator_strings_up_to_order(3, separate_orders=true))
display(multi_spin_gen_operator_strings_up_to_order(3, separate_orders=false))


3-element Vector{Vector{String}}:
 ["-", "xi", "yi", "zi"]
 ["--", "+-", "xi-", "yi-", "zi-", "xi*xj", "xi*yj", "xi*zj", "yi*xj", "yi*yj", "yi*zj", "zi*xj", "zi*yj", "zi*zj"]
 ["---", "+--", "xi--", "xi+-", "yi--", "yi+-", "zi--", "zi+-", "xi*xj-", "xi*yj-"  …  "yi*zj*zk", "zi*xj*xk", "zi*xj*yk", "zi*xj*zk", "zi*yj*xk", "zi*yj*yk", "zi*yj*zk", "zi*zj*xk", "zi*zj*yk", "zi*zj*zk"]

62-element Vector{String}:
 "-"
 "xi"
 "yi"
 "zi"
 "--"
 "+-"
 "xi-"
 "yi-"
 "zi-"
 "xi*xj"
 ⋮
 "zi*xj*xk"
 "zi*xj*yk"
 "zi*xj*zk"
 "zi*yj*xk"
 "zi*yj*yk"
 "zi*yj*zk"
 "zi*zj*xk"
 "zi*zj*yk"
 "zi*zj*zk"

In [52]:
function multi_spin_gen_operators_of_order(n::Int)::Vector{Term}
    # create operator strings
    operator_strings::Vector{String} = multi_spin_gen_operator_strings_of_order(n)
    # create operators
    operators::Vector{Term} = []
    for op in operator_strings
        push!(operators, make_term(op))
    end
    return operators
end
# test
display(multi_spin_gen_operators_of_order(3))

44-element Vector{Term}:
  aaa 
  a†aa 
  aaxᵢ 
  a†axᵢ 
  aayᵢ 
  a†ayᵢ 
  aazᵢ 
  a†azᵢ 
  axᵢxⱼ 
  axᵢyⱼ 
 ⋮
  zᵢxⱼxₖ 
  zᵢxⱼyₖ 
  zᵢxⱼzₖ 
  zᵢyⱼxₖ 
  zᵢyⱼyₖ 
  zᵢyⱼzₖ 
  zᵢzⱼxₖ 
  zᵢzⱼyₖ 
  zᵢzⱼzₖ 

In [53]:
function multi_spin_gen_operators_up_to_order(n::Int)::Vector{Vector{Term}}
    # create operators
    operators::Vector{Vector{Term}} = [multi_spin_gen_operators_of_order(i) for i in 1:n]
    return operators
end
# test
display(multi_spin_gen_operators_up_to_order(3))

3-element Vector{Vector{Term}}:
 [Term("", "", "-", [1], 1.0 + 0.0im, String[]), Term("x", "i", "", Int[], 1.0 + 0.0im, String[]), Term("y", "i", "", Int[], 1.0 + 0.0im, String[]), Term("z", "i", "", Int[], 1.0 + 0.0im, String[])]
 [Term("", "", "--", [1, 1], 1.0 + 0.0im, String[]), Term("", "", "+-", [1, 1], 1.0 + 0.0im, String[]), Term("x", "i", "-", [1], 1.0 + 0.0im, String[]), Term("y", "i", "-", [1], 1.0 + 0.0im, String[]), Term("z", "i", "-", [1], 1.0 + 0.0im, String[]), Term("xx", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("xy", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("xz", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("yx", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("yy", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("yz", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("zx", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("zy", "ij", "", Int[], 1.0 + 0.0im, String[]), Term("zz", "ij", "", Int[], 1.0 + 0.0im, String[])]
 [Term("", "", "---", [1, 1, 1], 1.0 + 0.0im, Stri

In [55]:
function multi_spin_gen_DE_of_order(n::Int)::Vector{DE_Term_Multi}
    # create operator strings
    operators::Vector{Term} = multi_spin_gen_operators_of_order(n)
    # create DE operators
    DE_operators = Vector{DE_Term_Multi}(undef, length(operators))
    @threads for i in 1:length(operators) 
        DE_operators[i] = time_derivative_expectation_of_operator_multispin(operators[i])
    end
    return DE_operators
end
# test
display(multi_spin_gen_DE_of_order(3))

44-element Vector{DE_Term_Multi}:
 d/dt⟨aaa⟩ = ∑ᵢ - 3g_i/2(i⟨a²xᵢ⟩ + ⟨a²yᵢ⟩) + 3√̅κβ√̅κβ⟨a²⟩ - 3κ/2κ⟨a³⟩
 d/dt⟨a†aa⟩ = ∑ᵢg_i/2( - 2i⟨a†axᵢ⟩ - 2⟨a†ayᵢ⟩ + i⟨a²xᵢ⟩ - ⟨a²yᵢ⟩) + 2√̅κβ√̅κβ⟨a†a⟩ - 3κ/2κ⟨a†a²⟩ + √̅κβ*√̅κβ*⟨a²⟩
 d/dt⟨aaxᵢ⟩ = ∑ₕ̡̡₌ᵢ( - g_h(i⟨axₕxᵢ⟩ + ⟨ayₕxᵢ⟩)) + 2√̅κβ√̅κβ⟨axᵢ⟩ - κκ⟨a²xᵢ⟩ + ig_i(⟨a³zᵢ⟩ - ⟨a⟩ - ⟨a†a²zᵢ⟩ - ⟨azᵢ⟩) - Δ_iΔ_i⟨a²yᵢ⟩ - 2γγ⟨a²xᵢ⟩ - Γ/2Γ⟨a²xᵢ⟩
 d/dt⟨a†axᵢ⟩ = ∑ₕ̡̡₌ᵢ(g_h/2( - i⟨a†xₕxᵢ⟩ - ⟨a†yₕxᵢ⟩ + i⟨axₕxᵢ⟩ - ⟨ayₕxᵢ⟩)) + √̅κβ√̅κβ⟨a†xᵢ⟩ - κκ⟨a†axᵢ⟩ + ig_i/2(2⟨a†a²zᵢ⟩ - ⟨a†⟩ - 2⟨a†²azᵢ⟩ - ⟨a†zᵢ⟩ + ⟨a⟩ + ⟨azᵢ⟩) - Δ_iΔ_i⟨a†ayᵢ⟩ - 2γγ⟨a†axᵢ⟩ - Γ/2Γ⟨a†axᵢ⟩ + √̅κβ*√̅κβ*⟨axᵢ⟩
 d/dt⟨aayᵢ⟩ = ∑ₕ̡̡₌ᵢ( - g_h(i⟨axₕyᵢ⟩ + ⟨ayₕyᵢ⟩)) + 2√̅κβ√̅κβ⟨ayᵢ⟩ - κκ⟨a²yᵢ⟩ - g_i(⟨a³zᵢ⟩ + ⟨a†a²zᵢ⟩ + ⟨azᵢ⟩ + ⟨a⟩) + Δ_iΔ_i⟨a²xᵢ⟩ - 2γγ⟨a²yᵢ⟩ - Γ/2Γ⟨a²yᵢ⟩
 d/dt⟨a†ayᵢ⟩ = ∑ₕ̡̡₌ᵢ(g_h/2( - i⟨a†xₕyᵢ⟩ - ⟨a†yₕyᵢ⟩ + i⟨axₕyᵢ⟩ - ⟨ayₕyᵢ⟩)) + √̅κβ√̅κβ⟨a†yᵢ⟩ - κκ⟨a†ayᵢ⟩ - g_i/2(2⟨a†a²zᵢ⟩ + 2⟨a†²azᵢ⟩ + ⟨a†zᵢ⟩ + ⟨a†⟩ + ⟨azᵢ⟩ + ⟨a⟩) + Δ_iΔ_i⟨a†axᵢ⟩ - 2γγ⟨a†ayᵢ⟩ - Γ/2Γ⟨a†ayᵢ⟩ + √̅κβ*√̅κβ*⟨ayᵢ⟩
 d/dt⟨aazᵢ⟩ = ∑

In [59]:
function multi_spin_gen_DE_up_to_order(n::Int)::Vector{Vector{DE_Term_Multi}}
    # create DE operators
    DE_operators::Vector{Vector{DE_Term_Multi}} = Vector{Vector{DE_Term_Multi}}(undef, n)
    for i in 1:n
        DE_operators[i] = multi_spin_gen_DE_of_order(i)
    end
    return DE_operators
end
# test
eq_of_eq = multi_spin_gen_DE_up_to_order(2)
print("done")
for eqs in eq_of_eq
    for eq in eqs
        display(eq)
    end
end

d/dt⟨a⟩ = ∑ᵢ - g_i/2(i⟨xᵢ⟩ + ⟨yᵢ⟩) + √̅κβ - κ/2κ⟨a⟩

d/dt⟨xᵢ⟩ = ig_i(⟨azᵢ⟩ - ⟨a†zᵢ⟩) - Δ_iΔ_i⟨yᵢ⟩ - 2γγ⟨xᵢ⟩ - Γ/2Γ⟨xᵢ⟩

d/dt⟨yᵢ⟩ =  - g_i(⟨azᵢ⟩ + ⟨a†zᵢ⟩) + Δ_iΔ_i⟨xᵢ⟩ - 2γγ⟨yᵢ⟩ - Γ/2Γ⟨yᵢ⟩

d/dt⟨zᵢ⟩ = g_i(⟨ayᵢ⟩ - i⟨axᵢ⟩ + ⟨a†yᵢ⟩ + i⟨a†xᵢ⟩) - Γ(⟨zᵢ⟩ + 1)

d/dt⟨aa⟩ = ∑ᵢ - g_i(i⟨axᵢ⟩ + ⟨ayᵢ⟩) + 2√̅κβ√̅κβ⟨a⟩ - κκ⟨a²⟩

...

d/dt⟨yᵢzⱼ⟩ =  - g_i(⟨azᵢzⱼ⟩ + ⟨a†zᵢzⱼ⟩) + Δ_iΔ_i⟨xᵢzⱼ⟩ - 2γγ⟨yᵢzⱼ⟩ + g_j(⟨ayᵢyⱼ⟩ - i⟨ayᵢxⱼ⟩ + ⟨a†yᵢyⱼ⟩ + i⟨a†yᵢxⱼ⟩) + Γ( - 3/2⟨yᵢzⱼ⟩ - ⟨yᵢ⟩)

d/dt⟨zᵢxⱼ⟩ =  - Δ_jΔ_j⟨zᵢyⱼ⟩ + g_i(⟨ayᵢxⱼ⟩ - i⟨axᵢxⱼ⟩ + ⟨a†yᵢxⱼ⟩ + i⟨a†xᵢxⱼ⟩) - 2γγ⟨zᵢxⱼ⟩ + ig_j(⟨azᵢzⱼ⟩ - ⟨a†zᵢzⱼ⟩) + Γ( - 3/2⟨zᵢxⱼ⟩ - ⟨xⱼ⟩)

d/dt⟨zᵢyⱼ⟩ = Δ_jΔ_j⟨zᵢxⱼ⟩ + g_i(⟨ayᵢyⱼ⟩ - i⟨axᵢyⱼ⟩ + ⟨a†yᵢyⱼ⟩ + i⟨a†xᵢyⱼ⟩) - 2γγ⟨zᵢyⱼ⟩ - g_j(⟨azᵢzⱼ⟩ + ⟨a†zᵢzⱼ⟩) + Γ( - 3/2⟨zᵢyⱼ⟩ - ⟨yⱼ⟩)

d/dt⟨zᵢzⱼ⟩ = g_i(⟨ayᵢzⱼ⟩ - i⟨axᵢzⱼ⟩ + ⟨a†yᵢzⱼ⟩ + i⟨a†xᵢzⱼ⟩) + g_j(⟨azᵢyⱼ⟩ - i⟨azᵢxⱼ⟩ + ⟨a†zᵢyⱼ⟩ + i⟨a†zᵢxⱼ⟩) - Γ(2⟨zᵢzⱼ⟩ + ⟨zⱼ⟩ + ⟨zᵢ⟩)

done

In [ ]:
################################################################
#### Complexity analysis #######################################

In [35]:
# Create table of number of terms in each order
for i in 1:5
    println(i, "  & ", length(single_spin_gen_operator_strings_of_order(i)) , "  & ", length(multi_spin_gen_operator_strings_of_order(i)), "  \\\\")
end

1  & 4  & 4  \\
2  & 5  & 14  \\
3  & 8  & 44  \\
4  & 9  & 135  \\
5  & 12  & 408  \\


In [48]:
# Number of spin index combinations by number of spins  (only increasing spin indexings, ie. 222, 223, 224, 225, 233, 234, 235, 244, 245, 255, but not 232 or 322)
# and number of spin operators
# calculate via spin_index_combinations(number_of_spins, number_of_spin_operators)
# make a table
numbers_of_spins = 1:20
numbers_of_spin_operators = 1:4
# number of spins by row and number of operators by column
println(raw"\begin{tabular}{r "*"c "^length(numbers_of_spin_operators) *" }")
println(raw"\phantom{\# Spins} & \multicolumn{", length(numbers_of_spin_operators), raw"}{c}{\# Spin Operators} \\\\")
println(raw" \# Spins & ", join(numbers_of_spin_operators, " & "), raw" \\\\")
println(raw"\hline")
for n_spin in numbers_of_spins
    str = string(n_spin, " & ")
    for (i, n_op) in enumerate(numbers_of_spin_operators)
        str *= string(length(spin_index_combinations(n_spin, n_op))) 
        if i < length(numbers_of_spin_operators)
            str *= " & "
        end
    end
    println(str, " \\\\")
end
println(raw"\hline")
println(raw"\end{tabular}")

\begin{tabular}{r c c c c  }
\phantom{\# Spins} & \multicolumn{4}{c}{\# Spin Operators} \\
 \# Spins & 1 & 2 & 3 & 4 \\
\hline
1 & 1 & 0 & 0 & 0 \\
2 & 2 & 1 & 0 & 0 \\
3 & 3 & 3 & 1 & 0 \\
4 & 4 & 6 & 4 & 1 \\
5 & 5 & 10 & 10 & 5 \\
6 & 6 & 15 & 20 & 15 \\
7 & 7 & 21 & 35 & 35 \\
8 & 8 & 28 & 56 & 70 \\
9 & 9 & 36 & 84 & 126 \\
10 & 10 & 45 & 120 & 210 \\
11 & 11 & 55 & 165 & 330 \\
12 & 12 & 66 & 220 & 495 \\
13 & 13 & 78 & 286 & 715 \\
14 & 14 & 91 & 364 & 1001 \\
15 & 15 & 105 & 455 & 1365 \\
16 & 16 & 120 & 560 & 1820 \\
17 & 17 & 136 & 680 & 2380 \\
18 & 18 & 153 & 816 & 3060 \\
19 & 19 & 171 & 969 & 3876 \\
20 & 20 & 190 & 1140 & 4845 \\
\hline
\end{tabular}
